#### Pruebas Iniciales Cloud

In [ ]:
%%time
import duckdb
import pandas as pd

##### Mirando el Contenido de los Datasets en el Bucket

In [ ]:
# dataset pequeño
df = pd.read_csv('/home/ds/buckets/b1/datasets/dataset_pequeno.csv')
df.shape

In [ ]:
con = duckdb.connect()
DATA = '/home/ds/buckets/b1/datasets/'

# Columnas y tipos, sin cargar nada
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{DATA}gerencial_competencia_2026.csv.gz')")   # gerencial

In [ ]:
con.sql(f"SELECT count(*) FROM read_csv_auto('{DATA}gerencial_competencia_2026.csv.gz')")

In [ ]:
# los otros tres
for ds in ['gerencial', 'analistajr', 'analistasr']:
    n = con.sql(f"""
        SELECT count(*) 
        FROM (DESCRIBE SELECT * FROM read_csv_auto('{DATA}{ds}_competencia_2026.csv.gz'))
    """).fetchone()[0]
    print(f'{ds}: {n} columnas')

In [ ]:
con.sql(f"""
    SELECT foto_mes, count(*) as filas
    FROM read_csv_auto('{DATA}analistasr_competencia_2026.csv.gz')
    GROUP BY foto_mes ORDER BY foto_mes
""")

In [ ]:
con.sql(f"""
    SELECT foto_mes, clase_ternaria, count(*) as filas
    FROM read_csv_auto('{DATA}analistasr_competencia_2026.csv.gz')
    WHERE foto_mes >= 202106
    GROUP BY foto_mes, clase_ternaria
    ORDER BY foto_mes, clase_ternaria
""")

In [ ]:
con = duckdb.connect()

con.sql("""
    SELECT foto_mes, count(*) as filas
    FROM read_csv_auto('/home/ds/buckets/b1/datasets/dataset_pequeno.csv')
    GROUP BY foto_mes ORDER BY foto_mes
""")

In [ ]:
import duckdb
con = duckdb.connect()

ORIGEN = '/home/ds/buckets/b1/datasets/analistasr_competencia_2026.csv.gz'
DESTINO = '/home/ds/buckets/b1/prueba/data/prueba_4meses.parquet'

con.execute(f"""
    COPY (
        SELECT * FROM read_csv_auto('{ORIGEN}')
        WHERE foto_mes IN (202103, 202105, 202106, 202108)
    ) TO '{DESTINO}' (FORMAT PARQUET)
""")

print('listo')

In [ ]:
%%time
import time
con = duckdb.connect()

ORIGEN = '/home/ds/buckets/b1/datasets/analistasr_competencia_2026.csv.gz'
DESTINO = '/home/ds/buckets/b1/01_Prueba/data/prueba_4meses.parquet'

con.execute(f"""
    COPY (
        SELECT * FROM read_csv_auto('{ORIGEN}')
        WHERE foto_mes IN (202103, 202105, 202106, 202108)
    ) TO '{DESTINO}' (FORMAT PARQUET)
""")

print('listo')

In [ ]:
con.sql(f"""
    SELECT foto_mes, count(*) as filas
    FROM read_parquet('{DESTINO}')
    GROUP BY foto_mes ORDER BY foto_mes
""")

In [ ]:
con.sql(f"""
    SELECT foto_mes, clase_ternaria, count(*) as filas
    FROM read_parquet('{DESTINO}')
    GROUP BY foto_mes, clase_ternaria
    ORDER BY foto_mes, clase_ternaria
""")

In [ ]:
con.sql(f"""
    SELECT foto_mes,
           count(*) as filas,
           count(ccajas_depositos) as no_nulos,
           avg(ccajas_depositos) as promedio
    FROM read_parquet('{DESTINO}')
    GROUP BY foto_mes ORDER BY foto_mes
""")